# VodHunter Backfill Ingest (Colab)

This notebook runs the repo's backfill ingest flow for a single Twitch streamer across a configurable number of past days.

Use a GPU runtime if available: `Runtime -> Change runtime type -> GPU`.

Notes:
- Your Postgres database with `pgvector` must be reachable from Colab.
- Twitch API credentials are required.
- This notebook calls the existing `run_backfill_ingest()` Python entrypoint directly.
- If the Colab session disconnects, the job stops. Resume behavior depends on the DB-backed ingest state already implemented by the repo.


In [ ]:
# Optional: clone the repo when running in a fresh Colab session.
# Set REPO_URL to your Git remote before running this cell.

REPO_URL = "https://github.com/manofshad/VodHunter"
REPO_DIR = "/content/VodHunter"

if REPO_URL:
    import os
    if not os.path.exists(REPO_DIR):
        !git clone "$REPO_URL" "$REPO_DIR"
    %cd "$REPO_DIR"
else:
    print("Set REPO_URL if you want the notebook to clone the repository automatically.")


In [ ]:
# Install system and Python dependencies.
!apt-get update -y
!apt-get install -y ffmpeg
!pip install -U pip
!pip install -r backend/requirements.txt
!pip install yt-dlp ipywidgets


In [ ]:
# Runtime configuration.
# Fill these in before running the rest of the notebook.

STREAMER = ""
DAYS = 7
DATABASE_URL = ""
TWITCH_CLIENT_ID = ""
TWITCH_CLIENT_SECRET = ""

# Optional tuning.
INGEST_CHUNK_SECONDS = "60"

assert STREAMER.strip(), "Set STREAMER first"
assert int(DAYS) >= 1, "DAYS must be >= 1"
assert DATABASE_URL.strip(), "Set DATABASE_URL first"
assert TWITCH_CLIENT_ID.strip(), "Set TWITCH_CLIENT_ID first"
assert TWITCH_CLIENT_SECRET.strip(), "Set TWITCH_CLIENT_SECRET first"


In [ ]:
# Export environment variables used by the repo.
import os
import sys
from pathlib import Path

os.environ["DATABASE_URL"] = DATABASE_URL.strip()
os.environ["TWITCH_CLIENT_ID"] = TWITCH_CLIENT_ID.strip()
os.environ["TWITCH_CLIENT_SECRET"] = TWITCH_CLIENT_SECRET.strip()
os.environ["INGEST_CHUNK_SECONDS"] = INGEST_CHUNK_SECONDS.strip()

repo_root = Path.cwd()
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

print(f"Repo root: {repo_root}")


In [ ]:
# Bootstrap imports and verify CUDA visibility.
import torch
from runners.run_backfill_ingest import run_backfill_ingest

print("CUDA available:", torch.cuda.is_available())
print("Backfill runner ready")


In [ ]:
# Run backfill ingest and stream log lines inline.
logs = []

def out(message: str) -> None:
    logs.append(str(message))
    print(message)

result = run_backfill_ingest(STREAMER, int(DAYS), out=out)
print(result)


In [ ]:
# Summarize the result and show recent log lines.
TAIL_LINES = 25

print("ingested:", result.ingested)
print("resumed:", result.resumed)
print("skipped:", result.skipped)
print("failed:", result.failed)
print("recent logs:")
for line in logs[-TAIL_LINES:]:
    print(line)
